In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from tqdm.auto import tqdm

DATA_PATH = "Data/crsp_ff3_daily_1995_2024.parquet"


/Users/nikolauswieland/Documents/masters_thesis/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(13748674, 11)


,permno,date,ret,prc,vol,shrout,mktrf,smb,hml,rf,excess_ret
0,10026,1995-01-03,-0.010753,11.5,18872.0,9408.0,-0.0026,-0.0097,0.0094,0.0002,-0.010953
1,10026,1995-01-04,0.0,11.5,33900.0,9408.0,0.0032,-0.0029,0.0047,0.0002,-0.0002
2,10026,1995-01-05,-0.01087,11.375,6202.0,9408.0,-0.0005,0.0004,0.0005,0.0002,-0.01107
3,10026,1995-01-06,0.010989,11.5,7510.0,9408.0,0.0018,-0.0001,0.0013,0.0002,0.010789
4,10026,1995-01-09,-0.01087,11.375,4000.0,9408.0,0.0008,0.0002,0.0013,0.0002,-0.01107


In [ ]:
df = pd.read_parquet(DATA_PATH)
df["date"] = pd.to_datetime(df["date"])

needed = {"date", "permno", "excess_ret", "mktrf", "smb", "hml"}
missing = needed - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

df = df.dropna(subset=["excess_ret", "mktrf", "smb", "hml"]).copy()
print(df.shape)
df.head()

In [2]:
TRAIN_END = "2023-12-31"
TEST_START, TEST_END = "2024-01-01", "2024-12-31"

train = df[df["date"] <= TRAIN_END].copy()
test  = df[(df["date"] >= TEST_START) & (df["date"] <= TEST_END)].copy()

print("Train:", train.shape)
print("Test :", test.shape)

#Enforcing minimum training history to the same as for other models
MIN_TRAIN_OBS = 1000

train_counts = train.groupby("permno").size()
eligible_permnos = train_counts[train_counts >= MIN_TRAIN_OBS].index.values

train = train[train["permno"].isin(eligible_permnos)].copy()
test  = test[test["permno"].isin(eligible_permnos)].copy()

print("Eligible permnos:", len(eligible_permnos))
print("Train (eligible):", train.shape)
print("Test  (eligible):", test.shape)

Train: (13064388, 11)
Test : (684286, 11)
Eligible permnos: 2716
Train (eligible): (13064388, 11)
Test  (eligible): (684286, 11)


In [ ]:
# Per-stock historical mean estimated on train
mu = train.groupby("permno")["excess_ret"].mean().rename("mean_pred")

# Attach to test
test_mean = test.merge(mu, on="permno", how="left")
test_mean = test_mean.dropna(subset=["mean_pred"]).copy()

# MSPE 
mspe_mean = np.mean((test_mean["excess_ret"] - test_mean["mean_pred"])**2)

print("Historical mean MSPE:", mspe_mean)

Historical mean MSPE: 0.002056709989691306


In [4]:
features = ["mktrf", "smb", "hml"]

def fit_ff3(stock_train: pd.DataFrame):
    X = sm.add_constant(stock_train[features])
    y = stock_train["excess_ret"]
    return sm.OLS(y, X).fit()

ff3_preds = []

permnos = np.sort(test_mean["permno"].unique())
for permno in tqdm(permnos, desc="FF3 per-stock OLS"):
    stock_train = train[train["permno"] == permno]
    stock_test  = test_mean[test_mean["permno"] == permno]  # same universe as mean

    if len(stock_train) < 50 or stock_test.empty:
        continue

    model = fit_ff3(stock_train)
    X_test = sm.add_constant(stock_test[features])
    pred = model.predict(X_test).values

    tmp = stock_test[["date", "permno", "excess_ret", "mean_pred"]].copy()
    tmp["ff3_pred"] = pred
    ff3_preds.append(tmp)

preds = pd.concat(ff3_preds, ignore_index=True)
preds.head()
mspe_ff3 = np.mean((preds["excess_ret"] - preds["ff3_pred"])**2)
print("FF3 MSPE:", mspe_ff3)

FF3 per-stock OLS: 100%|██████████| 2716/2716 [00:22<00:00, 123.29it/s]


FF3 MSPE: 0.001920165841881584


In [5]:
ratio = mspe_ff3 / mspe_mean
r2_oos_vs_mean = 1 - ratio
improvement_pct = (mspe_mean - mspe_ff3) / mspe_mean * 100

print("MSPE Mean:", mspe_mean)
print("MSPE FF3 :", mspe_ff3)
print(f"MSPE ratio (FF3/Mean): {ratio:.6f}")
print(f"OOS R^2 vs Mean      : {r2_oos_vs_mean:.6f}")
print(f"Relative improvement : {improvement_pct:.4f}%")

MSPE Mean: 0.002056709989691306
MSPE FF3 : 0.001920165841881584
MSPE ratio (FF3/Mean): 0.933610
OOS R^2 vs Mean      : 0.066390
Relative improvement : 6.6390%
